# Satellite Image Exploration and Geospatial Preprocessing

This notebook reproduces the geospatial preprocessing workflow described in Fractal AI's **“Understanding Satellite Image for Geo-spatial Deep Learning”** article using the Open Cities AI Challenge `33cae6` scene.

The objective is to convert:

```text
GeoTIFF aerial imagery + GeoJSON building footprints
                    ↓
          aligned image/mask tiles
```

The final output of this notebook is used by `02_building_segmentation_unet.ipynb` for semantic-segmentation training.

**Scene:** `33cae6`  
**Target resolution:** `0.1 m/pixel`  
**Tile size:** `1024 × 1024`  
**Mask:** `0 = background`, `1 = building`


## 1. Environment and project paths

The notebook is expected to run from the repository's `notebooks/` directory. Raw data is kept separate from processed rasters and model-ready tiles.

Large raster data is intentionally excluded from GitHub.


In [ ]:
from pathlib import Path
import json
import math
import subprocess

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window

PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGE_TILE_DIR = PROJECT_ROOT / "data" / "images"
MASK_TILE_DIR = PROJECT_ROOT / "data" / "masks"

SOURCE_IMAGE = RAW_DIR / "33cae6.tif"
SOURCE_GEOJSON = RAW_DIR / "33cae6.geojson"

RESAMPLED_IMAGE = PROCESSED_DIR / "33cae6_0.1m.tif"
BUILDING_MASK = PROCESSED_DIR / "33cae6_mask.tif"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_TILE_DIR.mkdir(parents=True, exist_ok=True)
MASK_TILE_DIR.mkdir(parents=True, exist_ok=True)

assert SOURCE_IMAGE.exists(), f"Missing: {SOURCE_IMAGE}"
assert SOURCE_GEOJSON.exists(), f"Missing: {SOURCE_GEOJSON}"

print("Source image:", SOURCE_IMAGE)
print("Annotations :", SOURCE_GEOJSON)


## 2. Inspect the GeoJSON annotations

GeoJSON stores the building footprints as **vector polygons**. Before rasterization, we first confirm that the file is a FeatureCollection and inspect the number of annotated buildings.


In [ ]:
with SOURCE_GEOJSON.open("r", encoding="utf-8") as f:
    geojson = json.load(f)

print("Type:", geojson["type"])
print("Number of features:", len(geojson["features"]))
print("CRS metadata:", geojson.get("crs"))

assert geojson["type"] == "FeatureCollection"
assert len(geojson["features"]) == 4439


## 3. Inspect the source GeoTIFF

A GeoTIFF contains both pixel values and geospatial metadata.

Important properties include:

- **width / height:** raster dimensions in pixels,
- **bands:** image channels,
- **CRS:** coordinate reference system,
- **resolution:** ground distance represented by one pixel,
- **bounds:** geographic extent,
- **transform:** mapping between pixel coordinates and map coordinates.


In [ ]:
with rasterio.open(SOURCE_IMAGE) as src:
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)

    assert src.width == 37113
    assert src.height == 34306
    assert src.count == 4
    assert str(src.crs) == "EPSG:32737"


## 4. Resample the imagery to 0.1 m/pixel

The original raster has a resolution of approximately **0.07748 m/pixel**. Following the reference workflow, GDAL resamples it to **0.1 m/pixel**.

Bilinear interpolation is used for the continuous image values. The `-overwrite` option makes this cell safe to rerun without manually deleting the previous output.


In [ ]:
TARGET_RESOLUTION = 0.1

command = [
    "gdalwarp",
    "-overwrite",
    "-co", "COMPRESS=JPEG",
    "-co", "TILED=YES",
    "-co", "NUM_THREADS=ALL_CPUS",
    "-r", "bilinear",
    "-tr", str(TARGET_RESOLUTION), str(TARGET_RESOLUTION),
    str(SOURCE_IMAGE),
    str(RESAMPLED_IMAGE),
]

print(" ".join(command))
subprocess.run(command, check=True)


## 5. Verify the resampled raster

The resized image should retain the original CRS while changing its pixel dimensions and resolution.


In [ ]:
with rasterio.open(RESAMPLED_IMAGE) as img:
    print("Width:", img.width)
    print("Height:", img.height)
    print("Bands:", img.count)
    print("CRS:", img.crs)
    print("Resolution:", img.res)

    assert img.width == 28755
    assert img.height == 26580
    assert img.count == 4
    assert str(img.crs) == "EPSG:32737"
    assert np.allclose(img.res, (0.1, 0.1))


## 6. Visualize the RGB imagery

Rasterio returns arrays in `[bands, height, width]` order, while Matplotlib expects `[height, width, channels]`.

Only bands 1–3 are displayed as RGB. A reduced-size preview is read so the entire large raster does not need to be loaded into memory for visualization.


In [ ]:
with rasterio.open(RESAMPLED_IMAGE) as img:
    preview = img.read(
        [1, 2, 3],
        out_shape=(3, 1000, 1000),
    )

plt.figure(figsize=(10, 10))
plt.imshow(preview.transpose(1, 2, 0))
plt.title("Resampled RGB imagery")
plt.axis("off")
plt.show()


## 7. Load and reproject the building footprints

The GeoJSON is read with GeoPandas. Its geometries must use the **same CRS as the raster** before rasterization; otherwise the polygons and image pixels would refer to different coordinate systems and the mask would be spatially misaligned.


In [ ]:
gdf = gpd.read_file(SOURCE_GEOJSON)

print("Number of buildings:", len(gdf))
print("Original CRS:", gdf.crs)

with rasterio.open(RESAMPLED_IMAGE) as img:
    image_crs = img.crs

gdf = gdf.to_crs(image_crs)

print("Reprojected CRS:", gdf.crs)

assert len(gdf) == 4439
assert gdf.crs == image_crs


## 8. Rasterize the polygons into a binary building mask

**Rasterization** converts vector building polygons into pixels.

Every pixel inside a building polygon receives value `1`; all other pixels receive `0`. The mask uses exactly the same dimensions and affine transform as the resampled image.


In [ ]:
shapes = [
    (geom, 1)
    for geom in gdf.geometry
    if geom is not None and not geom.is_empty
]

with rasterio.open(RESAMPLED_IMAGE) as img:
    mask = rasterize(
        shapes,
        out_shape=(img.height, img.width),
        transform=img.transform,
        fill=0,
        dtype="uint8",
    )

    mask_profile = img.profile.copy()

print("Mask shape:", mask.shape)
print("Mask dtype:", mask.dtype)
print("Mask values:", np.unique(mask))

assert mask.shape == (26580, 28755)
assert set(np.unique(mask)).issubset({0, 1})


## 9. Save the binary mask as a GeoTIFF

The mask is written with the same geospatial metadata as the resampled image. `NBITS=1` stores the logical binary raster efficiently while Rasterio still reads its values as `uint8`.


In [ ]:
mask_profile.update(
    driver="GTiff",
    count=1,
    dtype="uint8",
    compress="LZW",
    NBITS=1,
)

with rasterio.open(BUILDING_MASK, "w", **mask_profile) as dst:
    dst.write(mask, 1)

print("Saved mask to:", BUILDING_MASK)

with rasterio.open(RESAMPLED_IMAGE) as img, rasterio.open(BUILDING_MASK) as mask_src:
    assert img.width == mask_src.width
    assert img.height == mask_src.height
    assert img.crs == mask_src.crs
    assert img.transform == mask_src.transform

print("Image and mask geospatial metadata align.")


## 10. Preview the building mask

White pixels represent buildings and black pixels represent background. A downsampled view is used for efficient visualization.


In [ ]:
mask_preview = mask[::20, ::20]

plt.figure(figsize=(10, 10))
plt.imshow(mask_preview, cmap="gray")
plt.title("Rasterized building mask")
plt.axis("off")
plt.show()


## 11. Tile the image and mask into 1024×1024 patches

The full scene is far too large to use as one neural-network input. It is therefore divided into fixed-size **1024×1024 tiles**.

The image and mask are tiled using identical pixel windows. `boundless=True` pads the right and bottom edge tiles so that every output tile has the same dimensions.

Existing tiles with the same scene prefix are removed before regeneration so a rerun cannot leave stale files behind.


In [ ]:
TILE_SIZE = 1024
SCENE_ID = "33cae6"

def generate_tiles(src_path, output_dir, scene_id, tile_size=1024):
    output_dir.mkdir(parents=True, exist_ok=True)

    # Remove tiles from a previous run of this scene.
    for old_tile in output_dir.glob(f"{scene_id}_*.tif"):
        old_tile.unlink()

    tile_count = 0

    with rasterio.open(src_path) as src:
        for y in range(0, src.height, tile_size):
            for x in range(0, src.width, tile_size):
                window = Window(x, y, tile_size, tile_size)

                tile = src.read(
                    window=window,
                    boundless=True,
                    fill_value=0,
                )

                transform = src.window_transform(window)

                profile = src.profile.copy()
                profile.update(
                    width=tile_size,
                    height=tile_size,
                    transform=transform,
                )

                tile_path = output_dir / f"{scene_id}_{tile_count:05d}.tif"

                with rasterio.open(tile_path, "w", **profile) as dst:
                    dst.write(tile)

                tile_count += 1

    return tile_count

print("Tile size:", TILE_SIZE)


### 11.1 Generate image tiles


In [ ]:
image_tile_count = generate_tiles(
    RESAMPLED_IMAGE,
    IMAGE_TILE_DIR,
    SCENE_ID,
    TILE_SIZE,
)

print("Image tiles created:", image_tile_count)


### 11.2 Generate matching mask tiles

The binary mask is tiled with exactly the same grid and window size as the image.


In [ ]:
mask_tile_count = generate_tiles(
    BUILDING_MASK,
    MASK_TILE_DIR,
    SCENE_ID,
    TILE_SIZE,
)

print("Mask tiles created:", mask_tile_count)

assert image_tile_count == mask_tile_count == 754


## 12. Verify the model-ready tile dataset

Before model training, verify:

- exactly 754 image tiles and 754 masks exist,
- filenames match one-to-one,
- every tile is exactly 1024×1024,
- corresponding image/mask tiles preserve the same CRS and transform.


In [ ]:
image_files = sorted(IMAGE_TILE_DIR.glob(f"{SCENE_ID}_*.tif"))
mask_files = sorted(MASK_TILE_DIR.glob(f"{SCENE_ID}_*.tif"))

print("Image tiles:", len(image_files))
print("Mask tiles :", len(mask_files))

assert len(image_files) == len(mask_files) == 754
assert [p.name for p in image_files] == [p.name for p in mask_files]

for image_path, mask_path in zip(image_files, mask_files):
    with rasterio.open(image_path) as image_src, rasterio.open(mask_path) as mask_src:
        assert image_src.width == image_src.height == TILE_SIZE
        assert mask_src.width == mask_src.height == TILE_SIZE
        assert image_src.crs == mask_src.crs
        assert image_src.transform == mask_src.transform

print("All image/mask tile pairs passed alignment checks.")


## 13. Visualize an aligned image/mask pair

To make the sanity check informative, select the first tile whose mask contains at least one building rather than choosing a potentially empty tile at random.


In [ ]:
sample_image_path = None
sample_mask_path = None

for image_path, mask_path in zip(image_files, mask_files):
    with rasterio.open(mask_path) as src:
        candidate_mask = src.read(1)

    if candidate_mask.sum() > 0:
        sample_image_path = image_path
        sample_mask_path = mask_path
        break

assert sample_image_path is not None

with rasterio.open(sample_image_path) as src:
    image_tile = src.read([1, 2, 3])

with rasterio.open(sample_mask_path) as src:
    mask_tile = src.read(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image_tile.transpose(1, 2, 0))
axes[0].set_title(f"Satellite Image\n{sample_image_path.name}")
axes[0].axis("off")

axes[1].imshow(mask_tile, cmap="gray")
axes[1].set_title("Building Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## 14. Preprocessing output

The preprocessing stage produces:

```text
33cae6 GeoTIFF + building GeoJSON
              ↓
      resample to 0.1 m/pixel
              ↓
      rasterize building mask
              ↓
       1024×1024 tiling
              ↓
754 image tiles + 754 matching masks
```

These files form the input dataset for `02_building_segmentation_unet.ipynb`.

### Important scope

This notebook performs **data preparation only**. It does not train or evaluate a segmentation model. Model training, validation metrics, and predictions are handled separately in notebook `02`.
